# FallahTech RAG — T2 : Validation des Projections Série A

**Pipeline** : Données historiques NCT 2023-2025 (PDF) ✕ Projections Business Plan (Excel)  
**Objectif** : Tableau de cohérence — hypothèse / valeur projetée / valeur historique / écart / niveau de risque

## 1. Installation & Configuration

In [11]:
!pip install -q groq openpyxl pandas langchain-text-splitters langchain-core pymupdf
print("✅ Dépendances installées")

✅ Dépendances installées


In [40]:
import os, getpass
from groq import Groq

try:
    from google.colab import userdata
    os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
    print('✅ Clé chargée depuis Colab Secrets')
except:
    os.environ['GROQ_API_KEY'] = getpass.getpass('🔑 Entrez votre clé Groq API : ')
    print('✅ Clé saisie manuellement')

client = Groq(api_key=os.environ['GROQ_API_KEY'])

# Test rapide
resp = client.chat.completions.create(
    model='llama-3.3-70b-versatile',
    messages=[{'role': 'user', 'content': 'Répondre juste: 1+1=?'}],
    temperature=0, max_tokens=5
)
print(f'✅ Groq OK — réponse test : {resp.choices[0].message.content.strip()}')

✅ Clé chargée depuis Colab Secrets
✅ Groq OK — réponse test : 2


In [13]:
import pathlib, zipfile
from google.colab import drive

drive.mount('/content/drive')

DRIVE_DOSSIER = '/content/drive/MyDrive/rag_projet/fallahVFinal'  # ← adapter
DATA_DIR = pathlib.Path(DRIVE_DOSSIER)

if not DATA_DIR.exists():
    print(f'❌ Dossier introuvable : {DATA_DIR}')
else:
    for fpath in DATA_DIR.glob('*.zip'):
        dest_zip = DATA_DIR / fpath.stem
        if not dest_zip.exists():
            with zipfile.ZipFile(fpath) as z:
                z.extractall(dest_zip)
            print(f'📦 Extrait : {fpath.name}')

    all_files = [f for f in sorted(DATA_DIR.rglob('*')) if f.is_file()]
    for f in all_files:
        print(f'  [{f.suffix.upper():>5}] {f.relative_to(DATA_DIR)}  ({f.stat().st_size/1024:.0f} KB)')
    print(f'\n✅ Total : {len(all_files)} fichier(s)')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  [.IPYNB] Copie de FallahTech_RAG_T2_v_final.ipynb  (145 KB)
  [ .PDF] DataRoom_FallahTech_Professionnelle/DataRoom_FallahTech_PDF/0.0_Index_DataRoom.pdf  (246 KB)
  [ .PDF] DataRoom_FallahTech_Professionnelle/DataRoom_FallahTech_PDF/1_Juridique/1.1_Statuts_FallahTech.pdf  (267 KB)
  [ .PDF] DataRoom_FallahTech_Professionnelle/DataRoom_FallahTech_PDF/1_Juridique/1.2_Contrat_Cooperative_Type.pdf  (247 KB)
  [ .PDF] DataRoom_FallahTech_Professionnelle/DataRoom_FallahTech_PDF/2_Financier/2.1_Etats_Financiers_Historiques_NCT_2023_2025.pdf  (324 KB)
  [ .PDF] DataRoom_FallahTech_Professionnelle/DataRoom_FallahTech_PDF/3_Operationnel/3.1_Registre_Personnel.pdf  (265 KB)
  [ .PDF] DataRoom_FallahTech_Professionnelle/DataRoom_FallahTech_PDF/4_Commercial/4.1_Etude_Marche_Synthese.pdf  (270 KB)
  [.XLSX] FallahTech_BusinessPlan_Complet.xlsx  (13 KB)
  [.IPYNB] FallahT

## 2. Parsing des documents

In [ ]:
# ── Parsing PDF (rawdict fitz)
import fitz, re, pathlib

def pdf_to_markdown(chemin_pdf):
    doc  = fitz.open(str(chemin_pdf))
    nom  = pathlib.Path(chemin_pdf).name
    pages_md = []

    for num_page, page in enumerate(doc, 1):
        raw   = page.get_text('rawdict')
        chars = []
        for bloc in raw['blocks']:
            if bloc.get('type') != 0:
                continue
            for line in bloc['lines']:
                for span in line['spans']:
                    for char in span['chars']:
                        chars.append((round(char['origin'][1], 1),
                                      char['origin'][0],
                                      char['c']))
        if not chars:
            continue

        chars.sort(key=lambda t: (round(t[0]/3)*3, t[1]))
        lignes = {}
        for y0, x0, c in chars:
            y_key = round(y0 / 3) * 3
            if y_key not in lignes:
                lignes[y_key] = []
            lignes[y_key].append((x0, c))

        lignes_texte = []
        for y_key in sorted(lignes.keys()):
            pts = sorted(lignes[y_key], key=lambda t: t[0])
            col    = pts[0][1]
            x_prev = pts[0][0]
            colonnes = []
            for x0, c in pts[1:]:
                gap = x0 - x_prev
                if gap > 40:
                    colonnes.append(col.strip())
                    col = c
                elif gap > 20:
                    col += ' ' + c
                else:
                    col += c
                x_prev = x0
            colonnes.append(col.strip())
            colonnes = [c for c in colonnes if c]
            if len(colonnes) > 1:
                lignes_texte.append(' | '.join(colonnes))
            elif colonnes:
                lignes_texte.append(colonnes[0])

        pages_md.append(f'## {nom} — Page {num_page}\n\n' + '\n'.join(lignes_texte))

    doc.close()
    return '\n\n---\n\n'.join(pages_md)


# ── Parsing Excel ────────────────────────────────────────────────
import pandas as pd

def excel_to_markdown(chemin):
    xl  = pd.read_excel(str(chemin), sheet_name=None, header=None)
    nom = pathlib.Path(chemin).name
    feuilles_md = []
    for feuille, df in xl.items():
        df = df.dropna(how='all').reset_index(drop=True)
        if df.empty:
            continue
        lignes = [f'### {feuille}']
        for _, row in df.iterrows():
            vals = [str(v).strip() for v in row
                    if str(v).strip() not in ('', 'nan')]
            if not vals:
                continue
            if len(vals) == 1:
                lignes.append(f'#### {vals[0]}')
            else:
                lignes.append(' | '.join(vals))
        feuilles_md.append('\n'.join(lignes))
    return f'## {nom}\n\n' + '\n\n'.join(feuilles_md)

print('✅ Fonctions de parsing définies')

In [ ]:
import json

SAVE_PATH = DATA_DIR / 'parsed_docs.json'
tous_docs = {}

for fpath in sorted(DATA_DIR.rglob('*')):
    if 'processed_docs' in str(fpath) or 'chroma' in str(fpath):
        continue
    nom = fpath.name
    if fpath.suffix.lower() == '.pdf':
        print(f'📄 Parsing PDF : {nom}')
        tous_docs[nom] = {'texte': pdf_to_markdown(fpath), 'type_doc': 'pdf', 'chemin': str(fpath)}
    elif fpath.suffix.lower() in ('.xlsx', '.xls'):
        print(f'📊 Parsing Excel : {nom}')
        tous_docs[nom] = {'texte': excel_to_markdown(fpath), 'type_doc': 'excel', 'chemin': str(fpath)}

with open(SAVE_PATH, 'w', encoding='utf-8') as f:
    json.dump(tous_docs, f, ensure_ascii=False, indent=2)

print(f'\n✅ {len(tous_docs)} document(s) parsés et sauvegardés')
for nom, d in tous_docs.items():
    print(f'  {nom[:50]:50s} | {d["type_doc"]:5s} | {len(d["texte"])} chars')

📄 Parsing PDF : 0.0_Index_DataRoom.pdf


NameError: name 'pdf_to_markdown' is not defined

## 3. Chunking

In [ ]:
from langchain_core.documents import Document

# ── Chunker PDF : 1 page = 1 chunk (NON MODIFIÉ) ──────────────
def chunker_pdf(nom_fichier, texte, chemin):
    chunks   = []
    OVERLAP  = 5
    sections = texte.split('\n\n---\n\n')
    sections = [s.strip() for s in sections if len(s.strip()) >= 50]

    for i, section in enumerate(sections):
        premiere_ligne = section.split('\n')[0]
        titre = premiere_ligne.replace('##', '').strip()
        match = re.search(r'Page (\d+)', titre)
        page  = match.group(1) if match else '?'

        prefixe = ''
        if i > 0:
            prev_section = sections[i - 1]
            if nom_fichier in prev_section.split('\n')[0]:
                lignes_data = [l for l in prev_section.split('\n')
                               if l.strip() and not l.startswith('##')]
                if lignes_data:
                    prefixe = '[suite de la page précédente]\n' + '\n'.join(lignes_data[-OVERLAP:]) + '\n\n'

        chunks.append(Document(
            page_content=prefixe + section,
            metadata={
                'source'  : f'{nom_fichier}/page_{page}',
                'fichier' : nom_fichier,
                'section' : titre,
                'page'    : page,
                'chemin'  : chemin,
                'type_doc': 'pdf',
                'annees'  : '2023-2025',
            }
        ))
    return chunks


# ── Chunker Excel : 1 feuille = 1 chunk (NON MODIFIÉ) ─────────
def chunker_excel(nom_fichier, texte, chemin):
    chunks = []
    entete = texte.split('\n###')[0].strip()
    feuilles = re.split(r'\n(?=### )', texte)

    for feuille in feuilles:
        feuille = feuille.strip()
        if len(feuille) < 30:
            continue
        nom_feuille = feuille.split('\n')[0].replace('###', '').strip()
        if nom_feuille.startswith('##'):
            continue
        chunks.append(Document(
            page_content=f'{entete}\n\n{feuille}',
            metadata={
                'source'  : f'{nom_fichier}/{nom_feuille}',
                'fichier' : nom_fichier,
                'feuille' : nom_feuille,
                'chemin'  : chemin,
                'type_doc': 'excel',
                'annees'  : '2025-2029',
            }
        ))
    return chunks

print('✅ Chunkers définis')

✅ Chunkers définis


In [ ]:
import fitz, re, pathlib

CHUNKS_HIST_PATH = DATA_DIR / 'chunks_historique.json'
CHUNKS_PROJ_PATH = DATA_DIR / 'chunks_projections.json'

with open(SAVE_PATH, encoding='utf-8') as f:
    tous_docs = json.load(f)

docs_historique  = []
docs_projections = []

for nom, d in tous_docs.items():
    texte, type_doc, chemin = d['texte'], d['type_doc'], d['chemin']
    if type_doc == 'pdf':
        chunks = chunker_pdf(nom, texte, chemin)
        docs_historique.extend(chunks)
        print(f'📄 PDF {nom} → {len(chunks)} chunks')
    else:
        chunks = chunker_excel(nom, texte, chemin)
        docs_projections.extend(chunks)
        print(f'📊 Excel {nom} → {len(chunks)} chunks')

def docs_to_json(docs):
    return [{'page_content': d.page_content, 'metadata': d.metadata} for d in docs]

with open(CHUNKS_HIST_PATH, 'w', encoding='utf-8') as f:
    json.dump(docs_to_json(docs_historique), f, ensure_ascii=False, indent=2)
with open(CHUNKS_PROJ_PATH, 'w', encoding='utf-8') as f:
    json.dump(docs_to_json(docs_projections), f, ensure_ascii=False, indent=2)

print(f'\n✅ Chunks sauvegardés : {len(docs_historique)} HIST | {len(docs_projections)} PROJ')

📄 PDF 0.0_Index_DataRoom.pdf → 1 chunks
📄 PDF 1.1_Statuts_FallahTech.pdf → 1 chunks
📄 PDF 1.2_Contrat_Cooperative_Type.pdf → 1 chunks
📄 PDF 2.1_Etats_Financiers_Historiques_NCT_2023_2025.pdf → 12 chunks
📄 PDF 3.1_Registre_Personnel.pdf → 2 chunks
📄 PDF 4.1_Etude_Marche_Synthese.pdf → 2 chunks
📊 Excel FallahTech_BusinessPlan_Complet.xlsx → 6 chunks

✅ Chunks sauvegardés : 19 HIST | 6 PROJ


test chunking avec chonkie


In [ ]:
# Installe Chonkie avec support sémantique
!pip install "chonkie[semantic]"

In [ ]:
from chonkie import SemanticChunker
import json
from langchain_core.documents import Document
import pathlib

# ⚙️ Initialise le chunker sémantique
semantic_chunker = SemanticChunker(
    embedding_model="minishlab/potion-base-32M",  # modèle d'embeddings
    threshold=0.75,  # similarité sémantique
    chunk_size=512   # max tokens par chunk
)


DATA_DIR        = pathlib.Path('/content/drive/MyDrive/rag_projet/fallahVFinal')
SAVE_PATH       = DATA_DIR / 'parsed_docs.json'
CHUNKS_HIST_PATH= DATA_DIR / 'chunks_historique.json'
CHUNKS_PROJ_PATH= DATA_DIR / 'chunks_projections.json'

# --- Load parsed docs (structure JSON existante) ---
with open(SAVE_PATH, 'r', encoding='utf-8') as f:
    tous_docs = json.load(f)

docs_historique  = []
docs_projections = []

for nom, d in tous_docs.items():
    texte   = d['texte']
    t_doc   = d['type_doc']
    chemin  = d['chemin']

    # --- Chonker sémantique sur le texte complet d'un doc ---
    chunks = semantic_chunker.chunk(texte)

    # Convertit chaque chunk en Document LangChain
    doc_chunks = [
        Document(page_content=c.text, metadata={
            'source': nom,
            'type_doc': t_doc,
            'chemin': chemin,
            'chunk_index': i,
            'token_count': c.token_count
        })
        for i, c in enumerate(chunks)
    ]

    # Sépare historique vs projections
    if t_doc == 'pdf':
        docs_historique.extend(doc_chunks)
        print(f'📄 PDF {nom} → {len(doc_chunks)} chunks')
    else:
        docs_projections.extend(doc_chunks)
        print(f'📊 Excel {nom} → {len(doc_chunks)} chunks')

# --- Aperçu (extrait de quelques chunks) ---
print("\nExemples de chunks (PDF) :")
for d in docs_historique[:3]:
    print("----")
    print(d.page_content[:200], "...")

print("\nExemples de chunks (Excel) :")
for d in docs_projections[:3]:
    print("----")
    print(d.page_content[:200], "...")

# --- Sauvegarde JSON finale ---
def docs_to_json(docs):
    return [{'page_content': d.page_content, 'metadata': d.metadata} for d in docs]

with open(CHUNKS_HIST_PATH, 'w', encoding='utf-8') as f:
    json.dump(docs_to_json(docs_historique), f, ensure_ascii=False, indent=2)

with open(CHUNKS_PROJ_PATH, 'w', encoding='utf-8') as f:
    json.dump(docs_to_json(docs_projections), f, ensure_ascii=False, indent=2)

print("✅ Chunks sémantiques sauvegardés.")


In [14]:
!pip install -q langchain-chroma langchain-huggingface chromadb sentence-transformers
print('✅ OK')

✅ OK


In [41]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

DATA_DIR   = pathlib.Path('/content/drive/MyDrive/rag_projet/fallahVFinal')
CHROMA_DIR = str(DATA_DIR / 'chroma_db')

CHUNKS_HIST_PATH = DATA_DIR / 'chunks_historique.json'
CHUNKS_PROJ_PATH = DATA_DIR / 'chunks_projections.json'

def json_to_docs(path):
    with open(path, encoding='utf-8') as f:
        data = json.load(f)
    return [Document(page_content=d['page_content'], metadata=d['metadata']) for d in data]

docs_historique  = json_to_docs(CHUNKS_HIST_PATH)
docs_projections = json_to_docs(CHUNKS_PROJ_PATH)
print(f'✅ Chunks chargés : {len(docs_historique)} HIST | {len(docs_projections)} PROJ')

print('⏳ Chargement BGE-M3...')
embeddings = HuggingFaceEmbeddings(
    model_name='BAAI/bge-m3',
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)
print('✅ BGE-M3 prêt')

✅ Chunks chargés : 19 HIST | 6 PROJ
⏳ Chargement BGE-M3...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

✅ BGE-M3 prêt


In [ ]:
import os, shutil
from langchain_chroma import Chroma

if os.path.exists(CHROMA_DIR):
    shutil.rmtree(CHROMA_DIR)
os.makedirs(CHROMA_DIR, exist_ok=True)

print(f'⏳ Indexation {len(docs_historique)} chunks historique...')
index_hist = Chroma.from_documents(
    documents=docs_historique,
    embedding=embeddings,
    collection_name='fallahtech_historique',
    persist_directory=CHROMA_DIR + '/historique'
)

print(f'⏳ Indexation {len(docs_projections)} chunks projections...')
index_proj = Chroma.from_documents(
    documents=docs_projections,
    embedding=embeddings,
    collection_name='fallahtech_projections',
    persist_directory=CHROMA_DIR + '/projections'
)

print(f'✅ Index créés — HIST: {index_hist._collection.count()} | PROJ: {index_proj._collection.count()}')

⏳ Indexation 19 chunks historique...
⏳ Indexation 6 chunks projections...
✅ Index créés — HIST: 19 | PROJ: 6


In [42]:
# ── Charger les index existants (si déjà indexés) ──────────────
# Exécuter SEULEMENT si les index ChromaDB existent déjà sur Drive
# (évite de re-indexer à chaque session)

from langchain_chroma import Chroma

index_hist = Chroma(
    collection_name='fallahtech_historique',
    embedding_function=embeddings,
    persist_directory=CHROMA_DIR + '/historique'
)
index_proj = Chroma(
    collection_name='fallahtech_projections',
    embedding_function=embeddings,
    persist_directory=CHROMA_DIR + '/projections'
)

n_hist = index_hist._collection.count()
n_proj = index_proj._collection.count()
print(f'Index historique  : {n_hist} chunks')
print(f'Index projections : {n_proj} chunks')
if n_hist == 0 or n_proj == 0:
    print('⚠️  Collection vide — relancer la cellule d\'indexation ci-dessus.')

Index historique  : 19 chunks
Index projections : 6 chunks


## 5. Pipeline RAG T2

In [43]:
import re, json

# ── CONFIG ──────────────────────────────────────────────────────
USE_RERANK = False   # False = désactiver le reranking LLM


# ── Reranking LLM ───────────────────────────────────────────────

_RERANK_PROMPT = (
    "Filtre financier. QUESTION : {question}\n\n"
    "Garde UNIQUEMENT les indices des chunks utiles (chiffres, faits liés à la question).\n"
    "CHUNKS :\n{chunks_text}\n\n"
    "Réponds SEULEMENT avec ce JSON : {{\"kept\": [0, 1, 2]}}"
)

def _rerank(question: str, chunks: list) -> list:
    if len(chunks) <= 2:
        return chunks
    chunks_text = '\n---\n'.join(
        f'[{i}] {d.metadata.get("source","?")}\n{d.page_content[:300]}'
        for i, (d, _) in enumerate(chunks)
    )
    try:
        resp = client.chat.completions.create(
            model='llama-3.3-70b-versatile',
            messages=[{'role': 'user', 'content': _RERANK_PROMPT.format(
                question=question, chunks_text=chunks_text)}],
            temperature=0, max_tokens=80,
        )
        raw = re.sub(r'```(?:json)?|```', '', resp.choices[0].message.content).strip()
        # [CHANGEMENT 2] Accepte "kept" OU "kept_indices" pour robustesse
        parsed = json.loads(raw)
        kept = parsed.get('kept') or parsed.get('kept_indices', [])
        kept = [i for i in kept if isinstance(i, int) and 0 <= i < len(chunks)]
        return [chunks[i] for i in kept] if kept else chunks
    except Exception:
        return chunks  # Fallback : garder tout


# ── Retrieval dual ──────────────────────────────────────────────
def retrieve_dual(query: str, k_hist: int = 6, k_proj: int = 5,
                  min_hist: int = 3, score_threshold: float = 1.2,
                  use_rerank: bool = None) -> dict:
    if use_rerank is None:
        use_rerank = USE_RERANK

    raw_hist = index_hist.similarity_search_with_score(query, k=k_hist)
    raw_proj = index_proj.similarity_search_with_score(query, k=k_proj)

    filt_hist = [(d, s) for d, s in raw_hist if s <= score_threshold]
    filt_proj = [(d, s) for d, s in raw_proj if s <= score_threshold]

    # Garantie min_hist
    if len(filt_hist) < min_hist:
        seen = {id(d) for d, _ in filt_hist}
        for d, s in sorted(raw_hist, key=lambda x: x[1]):
            if id(d) not in seen:
                filt_hist.append((d, s))
                seen.add(id(d))
            if len(filt_hist) >= min_hist:
                break

    if use_rerank:
        filt_hist = _rerank(query, filt_hist)
        filt_proj = _rerank(query, filt_proj)
        # Reappliquer garantie après reranking
        if len(filt_hist) < min_hist:
            seen = {id(d) for d, _ in filt_hist}
            for d, s in sorted(raw_hist, key=lambda x: x[1]):
                if id(d) not in seen:
                    filt_hist.append((d, s))
                    seen.add(id(d))
                if len(filt_hist) >= min_hist:
                    break

    all_sorted = sorted(
        [(d, s, 'HIST') for d, s in filt_hist] +
        [(d, s, 'PROJ') for d, s in filt_proj],
        key=lambda x: x[1]
    )
    return {'hist': filt_hist, 'proj': filt_proj,
            'all_sorted': all_sorted, 'reranked': use_rerank}


def build_context(ret: dict) -> tuple:
    ctx_hist = '\n\n'.join(
        f'[HIST | score={s:.3f} | {d.metadata.get("source","?")}]\n{d.page_content}'
        for d, s in ret['hist']
    ) or 'Aucune donnée historique disponible.'
    ctx_proj = '\n\n'.join(
        f'[PROJ | score={s:.3f} | {d.metadata.get("source","?")}]\n{d.page_content}'
        for d, s in ret['proj']
    ) or 'Aucune projection disponible.'
    return ctx_hist, ctx_proj


print(f'✅ retrieve_dual défini — USE_RERANK={USE_RERANK}')

✅ retrieve_dual défini — USE_RERANK=False


In [44]:
# ── Parsing JSON robuste ─────────────────────────────────────────

def clean_json_output(raw: str) -> str:
    """Extrait et nettoie le JSON depuis la réponse LLM."""
    # Supprimer fences markdown
    cleaned = re.sub(r'```(?:json)?\s*', '', raw)
    cleaned = re.sub(r'```\s*$', '', cleaned, flags=re.MULTILINE).strip()
    # Extraire le bloc JSON principal
    start = cleaned.find('{')
    end   = cleaned.rfind('}')
    if start != -1 and end != -1:
        cleaned = cleaned[start:end+1]
    # Réparer trailing commas (ex: {"a":1,} ou [1,2,])
    cleaned = re.sub(r',\s*([}\]])', r'\1', cleaned)
    return cleaned.strip()


def safe_parse_json(raw: str) -> dict:
    """
    Parse JSON avec fallback multi-niveau.
    Retourne dict avec clé 'error' si échec total.
    """
    cleaned = clean_json_output(raw)
    # Tentative 1 : parse direct
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass
    # Tentative 2 : extraire juste le premier objet JSON valide
    try:
        # Cherche des blocs JSON imbriqués valides
        for match in re.finditer(r'\{[^{}]*\}', cleaned):
            try:
                return json.loads(match.group())
            except json.JSONDecodeError:
                continue
    except Exception:
        pass
    # Échec total : retourner structure d'erreur avec le raw pour debug
    return {'error': 'JSON parsing failed', 'raw': raw}


print('✅ clean_json_output et safe_parse_json définis')

✅ clean_json_output et safe_parse_json définis


In [45]:
# ── Système Prompt T2 ────────────────────────────────────────────


SYSTEM_PROMPT_T2 = """Tu es analyste financier senior — due diligence Série A FallahTech.

MISSION : Croiser projections Business Plan 2025-2029 avec historique NCT 2023-2025. Qualifier chaque hypothèse par niveau de risque.

RÈGLES
1. Utilise UNIQUEMENT les données des chunks [HIST] et [PROJ] fournis.
2. Valeur absente = "Non disponible". Relire tous les chunks avant de l'écrire.
3. Conflit PDF vs Excel → retenir PDF certifié NCT, signaler dans conflits[].
4. Citer source exacte pour chaque chiffre.

ANALYSE (dans l'ordre)
1. Extraire valeurs historiques 2023, 2024, 2025 selon le question ainsi que selon  l'historique disponible .
2. Calculer évolutions : (val_N - val_N-1) / val_N-1 × 100.
3. Extraire projections selon année demandé (2026, 2027, 2028, 2029) sinon utulise tout les annees de projection dispo
4. Calculer écart_pct = (val_projetee - val_2025) / val_2025 × 100 pour chaque année.
5. Classifier risque vs meilleure croissance historique :
   ≤ 1x → conservatrice | 1x–1.5x → realiste | 1.5x–2.5x → optimiste | >2.5x → tres_optimiste

EXEMPLE : Historique +212% puis +111%. Projection 2026 = +53% vs 2025 → conservatrice.

RETOURNER UNIQUEMENT ce JSON valide, sans texte avant ni après :
{"metrique": "...", "explication": "2-3 phrases.", "historique": {"2023": {"valeur": "X TND", "source": "..."}, "2024": {"valeur": "X TND", "source": "..."}, "2025": {"valeur": "X TND", "source": "..."}}, "evolutions_historiques": {"2023_2024": {"ecart_pct": "+X%", "ecart_absolu": "+X TND"}, "2024_2025": {"ecart_pct": "+X%", "ecart_absolu": "+X TND"}}, "tableau": [{"annee": 2026, "valeur_projetee": "X TND", "source_projection": "...", "valeur_historique_ref": "X TND (2025)", "ecart_pct": "+X%", "ecart_absolu": "+X TND", "taux_risque": "conservatrice", "explication": "..."}], "conflits": [], "sources": ["PDF/...", "Excel/..."]}"""

print('✅ SYSTEM_PROMPT_T2 défini')

✅ SYSTEM_PROMPT_T2 défini


In [46]:
def analyser_t2(question: str, k: int = 6, verbose: bool = False,
                use_rerank: bool = None) -> dict:
    """Pipeline RAG complet T2."""
    # 1. Retrieval
    ret = retrieve_dual(question, k_hist=k, k_proj=max(k-1, 4),
                        min_hist=3, use_rerank=use_rerank)
    ctx_hist, ctx_proj = build_context(ret)

    if verbose:
        print('--- CONTEXTE HIST ---')
        print(ctx_hist[:1200])
        print('--- CONTEXTE PROJ ---')
        print(ctx_proj[:600])

    # 2. Prompt utilisateur

    user_prompt = (
        f"QUESTION T2 : {question}\n\n"
        "=== DONNÉES HISTORIQUES NCT 2023-2025 (PDF certifié — priorité) ===\n"
        f"{ctx_hist}\n\n"
        "=== PROJECTIONS BUSINESS PLAN 2025-2029 (Excel) ===\n"
        f"{ctx_proj}\n\n"
        "→ Suivre les 5 étapes d'analyse. Répondre UNIQUEMENT en JSON valide."
    )

    # 3. Appel LLM
    response = client.chat.completions.create(
        model='llama-3.3-70b-versatile',
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT_T2},
            {'role': 'user',   'content': user_prompt},
        ],
        temperature=0.1,
        max_tokens=2500,
    )
    raw = response.choices[0].message.content

    # 4. Parsing JSON robuste

    parsed = safe_parse_json(raw)

    return {
        'question'     : question,
        'result'       : parsed,
        'raw_output'   : raw,          # [CHANGEMENT 7] Exposer raw pour debug
        'sources_hist' : [d.metadata.get('source', '?') for d, _ in ret['hist']],
        'sources_proj' : [d.metadata.get('source', '?') for d, _ in ret['proj']],
        'scores_hist'  : [round(s, 4) for _, s in ret['hist']],
        'scores_proj'  : [round(s, 4) for _, s in ret['proj']],
        'tokens'       : response.usage.total_tokens,
        'n_chunks_hist': len(ret['hist']),
        'n_chunks_proj': len(ret['proj']),
        'reranked'     : ret['reranked'],
    }


print('✅ analyser_t2 défini')

✅ analyser_t2 défini


## 6. Affichage

In [47]:
from IPython.display import display, HTML

_RISK_STYLE = {
    'conservatrice' : {'bg': '#d4edda', 'fg': '#155724', 'dot': '#28a745', 'label': 'Conservatrice'},
    'realiste'      : {'bg': '#cce5ff', 'fg': '#004085', 'dot': '#0066cc', 'label': 'Réaliste'},
    'optimiste'     : {'bg': '#fff3cd', 'fg': '#856404', 'dot': '#ffc107', 'label': 'Optimiste'},
    'tres_optimiste': {'bg': '#f8d7da', 'fg': '#721c24', 'dot': '#dc3545', 'label': 'Très optimiste'},
    'tres optimiste': {'bg': '#f8d7da', 'fg': '#721c24', 'dot': '#dc3545', 'label': 'Très optimiste'},
    'indetermine'   : {'bg': '#e2e3e5', 'fg': '#383d41', 'dot': '#6c757d', 'label': 'Indéterminé'},
}


def _risk_badge(taux_risque: str) -> str:
    r = _RISK_STYLE.get(str(taux_risque).lower().strip(),
                        {'dot': '#6c757d', 'fg': '#383d41', 'label': taux_risque})
    return (
        f"<span style='display:inline-flex;align-items:center;gap:5px;font-weight:600;"
        f"color:{r['fg']}'>"
        f"<span style='width:10px;height:10px;border-radius:50%;background:{r['dot']};"
        f"display:inline-block'></span>{r['label']}</span>"
    )


def display_t2(result: dict, show_meta: bool = True) -> None:
    res = result.get('result', {})

    if 'error' in res:
        display(HTML(
            f"<div style='color:#721c24;background:#f8d7da;padding:12px;border-radius:6px'>"
            f"<b>Erreur parsing JSON :</b> {res['error']}<br>"
            f"<pre style='font-size:0.8em;margin-top:8px'>{res.get('raw','')[:400]}</pre></div>"
        ))
        return

    metrique   = res.get('metrique', 'Métrique inconnue')
    explication= res.get('explication', '')
    historique = res.get('historique', {})
    evolutions = res.get('evolutions_historiques', {})
    tableau    = res.get('tableau', [])
    conflits   = res.get('conflits', [])

    html = (
        "<div style='font-family:Arial,sans-serif;max-width:980px'>"
        f"<h3 style='background:#1a252f;color:#ffffff;padding:10px 14px;"
        f"border-radius:6px 6px 0 0;margin:0'>Output — {metrique}</h3>"
        "<div style='border:1px solid #dee2e6;border-top:none;padding:14px 16px;"
        "border-radius:0 0 6px 6px'>"
    )

    def _h(y):
        h = historique.get(y, {})
        return h.get('valeur', 'Non disponible') if isinstance(h, dict) else str(h)
    def _e(k):
        e = evolutions.get(k, {})
        return e.get('ecart_pct', '?') if isinstance(e, dict) else '?'

    v23, v24, v25 = _h('2023'), _h('2024'), _h('2025')
    e2324, e2425  = _e('2023_2024'), _e('2024_2025')

    hist_prose = (
        f"Historique certifié NCT : 2023 = <b>{v23}</b>, 2024 = <b>{v24}</b> "
        f"({e2324} vs 2023), 2025 = <b>{v25}</b> ({e2425} vs 2024). "
    )
    if explication:
        hist_prose += explication

    html += f"<p style='font-size:0.90em;color:#fff;line-height:1.55;margin:0 0 14px'>{hist_prose}</p>"

    if tableau:
        html += (
            "<table style='border-collapse:collapse;width:100%;font-size:0.85em'>"
            "<thead><tr style='background:#1a252f;color:#f6f6f6'>"
            "<th style='padding:8px 10px;text-align:left'>Hypothèse</th>"
            "<th style='padding:8px 10px;text-align:left'>Valeur projetée</th>"
            "<th style='padding:8px 10px;text-align:left'>Valeur historique (réf.)</th>"
            "<th style='padding:8px 10px;text-align:left'>Écart</th>"
            "<th style='padding:8px 10px;text-align:left'>Niveau de risque</th>"
            "</tr></thead><tbody>"
        )
        for i, row in enumerate(tableau):
            annee    = row.get('annee', '?')
            val_proj = row.get('valeur_projetee') or row.get('valeur_projete') or '—'
            val_ref  = row.get('valeur_historique_ref') or row.get('valeur_historique_2025') or v25
            ecart    = row.get('ecart_pct', '—')
            taux     = str(row.get('taux_risque', 'indetermine')).lower().strip()
            hypoth   = str(row.get('explication', row.get('hypothese', '')))
            hypoth_s = hypoth[:120] + ('...' if len(hypoth) > 120 else '')
            badge    = _risk_badge(taux)
            bg_style = _RISK_STYLE.get(taux, {}).get('bg', '#fff')
            fg_style = _RISK_STYLE.get(taux, {}).get('fg', '#fff')
            row_bg   = bg_style if i % 2 == 0 else '#fff'
            html += (
                f"<tr style='background:{row_bg};border-bottom:1px solid #dee2e6'>"
                f"<td style='padding:8px 10px;color:#333;font-size:0.82em'>{metrique} {annee}<br>"
                f"<span style='color:#666;font-size:0.85em'>{hypoth_s}</span></td>"
                f"<td style='padding:8px 10px;font-weight:600'>{val_proj}</td>"
                f"<td style='padding:8px 10px'>{val_ref}</td>"
                f"<td style='padding:8px 10px;font-weight:700;color:{fg_style}'>{ecart}</td>"
                f"<td style='padding:8px 10px'>{badge}</td>"
                f"</tr>"
            )
        html += "</tbody></table>"
        html += (
            "<div style='font-size:0.78em;color:#ffffff;margin-top:6px'>"
            "<span style='color:#28a745'>●</span> Conservatrice &nbsp;"
            "<span style='color:#0066cc'>●</span> Réaliste &nbsp;"
            "<span style='color:#ffc107'>●</span> Optimiste &nbsp;"
            "<span style='color:#dc3545'>●</span> Très optimiste &nbsp;"
            "<span style='color:#6c757d'>●</span> Indéterminé"
            "</div>"
        )
    else:
        html += "<p style='color:#856404'>Aucun tableau de projections retourné.</p>"

    if conflits:
        html += (
            f"<div style='background:#fff3cd;border-left:4px solid #ffc107;"
            f"padding:10px 14px;margin-top:14px;border-radius:4px'>"
            f"<b style='color:#856404'>Conflits sources détectés ({len(conflits)})</b><ul style='margin:6px 0 0;font-size:0.85em'>"
        )
        for c in conflits:
            html += (
                f"<li><b>{c.get('metrique','?')} {c.get('annee','?')}</b> — "
                f"PDF: {c.get('valeur_pdf','?')} | Excel: {c.get('valeur_excel','?')}<br>"
                f"<em style='color:#ffffff'>{c.get('decision','')} — {c.get('explication','')}</em></li>"
            )
        html += "</ul></div>"

    if show_meta:
        rerank_s = 'oui' if result.get('reranked') else 'non'
        html += (
            "<details style='margin-top:10px;font-size:0.78em;color:#fff'>"
            "<summary style='cursor:pointer'>Métadonnées RAG</summary>"
            f"<p style='margin:4px 0'>Tokens: {result.get('tokens','?')} | "
            f"Chunks hist: {result.get('n_chunks_hist','?')} | "
            f"Chunks proj: {result.get('n_chunks_proj','?')} | "
            f"Reranking: {rerank_s}</p>"
            f"<p style='margin:2px 0'>Sources hist: {', '.join(result.get('sources_hist',[]))}</p>"
            f"<p style='margin:2px 0'>Sources proj: {', '.join(result.get('sources_proj',[]))}</p>"
            "</details>"
        )

    html += "</div></div>"
    display(HTML(html))


print('✅ display_t2 défini')

✅ display_t2 défini


## 7. Debug — Output brut avant parsing JSON

> **[AJOUT]** Affiche la réponse brute du modèle pour diagnostiquer les erreurs de parsing.

In [48]:
# ── Cellule de debug : afficher le raw output avant parsing ──────
# Décommenter et exécuter si un résultat retourne "erreur JSON parsing"

def debug_raw_output(question: str, k: int = 6) -> None:
    """
    Appelle le LLM et affiche :
    1. Le raw output (réponse brute du modèle)
    2. Le JSON nettoyé (après clean_json_output)
    3. Le résultat du parse (succès ou erreur détaillée)
    """
    ret = retrieve_dual(question, k_hist=k, k_proj=max(k-1, 4), min_hist=3, use_rerank=USE_RERANK)
    ctx_hist, ctx_proj = build_context(ret)

    user_prompt = (
        f"QUESTION T2 : {question}\n\n"
        "=== DONNÉES HISTORIQUES NCT 2023-2025 (PDF certifié — priorité) ===\n"
        f"{ctx_hist}\n\n"
        "=== PROJECTIONS BUSINESS PLAN 2025-2029 (Excel) ===\n"
        f"{ctx_proj}\n\n"
        "→ Suivre les 5 étapes d'analyse. Répondre UNIQUEMENT en JSON valide."
    )

    response = client.chat.completions.create(
        model='llama-3.3-70b-versatile',
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT_T2},
            {'role': 'user',   'content': user_prompt},
        ],
        temperature=0,
        max_tokens=2500,
    )
    raw = response.choices[0].message.content

    print('=' * 70)
    print('RAW OUTPUT DU MODÈLE (avant tout traitement) :')
    print('=' * 70)
    print(raw)
    print()

    cleaned = clean_json_output(raw)
    print('=' * 70)
    print('JSON NETTOYÉ (après clean_json_output) :')
    print('=' * 70)
    print(cleaned[:2000])
    print()

    print('=' * 70)
    print('RÉSULTAT DU PARSE :')
    print('=' * 70)
    try:
        parsed = json.loads(cleaned)
        print(f'✅ Parse réussi — clés : {list(parsed.keys())}')
    except json.JSONDecodeError as e:
        print(f'❌ Erreur : {e}')
        print(f'   Position : ligne {e.lineno}, col {e.colno}')
        # Afficher la zone problématique
        lines = cleaned.split('\n')
        start = max(0, e.lineno - 3)
        end   = min(len(lines), e.lineno + 2)
        print('   Contexte :')
        for i, l in enumerate(lines[start:end], start + 1):
            marker = '→' if i == e.lineno else ' '
            print(f'   {marker} {i:3d}: {l}')


# Exemple d'utilisation :
# debug_raw_output("Valide la projection du Chiffre d'Affaires FallahTech 2026-2029.")
print('✅ debug_raw_output défini — décommenter pour utiliser')

✅ debug_raw_output défini — décommenter pour utiliser


## 8. Évaluation

In [49]:
GROUND_TRUTH = {
    'CA': {
        'historique'  : {'2023': 250000, '2024': 780000, '2025': 1650000},
        'projections' : {'2026': 2520000, '2027': 3657000, '2028': 5118000, '2029': 6910000}
    },
    'Charges_Personnel': {
        'historique'  : {'2023': 180000, '2024': 320000, '2025': 480000},
        'projections' : {'2026': 600000, '2027': 750000, '2028': 937500}
    },
    'Marge_Brute_Pct': {
        'historique'  : {'2023':60, '2024': 64, '2025': 70},
        'projections' : {'2026': 70, '2027': 70, '2028': 70, '2029': 70}
    },
}

REQUIRED_FIELDS_T2 = [
    'metrique', 'explication', 'historique',
    'evolutions_historiques', 'tableau', 'sources'
]


def _num(s):
    try:
        return float(str(s).replace(' ', '').replace('TND', '').replace(',', '').replace('%', ''))
    except (ValueError, TypeError):
        return None


def test_faithfulness(result: dict, gt_key: str) -> dict:
    if gt_key not in GROUND_TRUTH:
        return {'status': 'N/A', 'errors': [], 'warnings': [f'Clé {gt_key} absente du GROUND_TRUTH']}

    gt  = GROUND_TRUTH[gt_key]
    res = result.get('result', {})
    errors, warnings = [], []

    for annee, expected in gt.get('historique', {}).items():
        h      = res.get('historique', {}).get(annee, {})
        actual = _num(h.get('valeur', '') if isinstance(h, dict) else h)
        if actual is None:
            warnings.append(f'Historique {annee} : valeur non parseable')
        elif abs(actual - expected) / max(abs(expected), 1) > 0.01:
            errors.append(f'Historique {annee} : attendu={expected:,.0f}, reçu={actual:,.0f}')

    for row in res.get('tableau', []):
        annee = str(row.get('annee', ''))
        if annee in gt.get('projections', {}):
            expected = gt['projections'][annee]
            actual   = _num(row.get('valeur_projetee') or row.get('valeur_projete'))
            if actual is None:
                warnings.append(f'Projection {annee} : valeur non parseable')
            elif abs(actual - expected) / max(abs(expected), 1) > 0.01:
                errors.append(f'Projection {annee} : attendu={expected:,.0f}, reçu={actual:,.0f}')

    # Vérifier cohérence des écarts déclarés
    hist_2025 = _num(
        res.get('historique', {}).get('2025', {}).get('valeur', '')
        if isinstance(res.get('historique', {}).get('2025'), dict) else ''
    )
    for row in res.get('tableau', []):
        vp   = _num(row.get('valeur_projetee') or row.get('valeur_projete'))
        ref  = _num(row.get('valeur_historique_ref') or row.get('valeur_historique_2025')) or hist_2025
        decl = _num(row.get('ecart_pct', '0'))
        if vp and ref and decl is not None and ref != 0:
            calc = ((vp - ref) / abs(ref)) * 100
            if abs(decl - calc) > 2.5:
                errors.append(f'Écart {row.get("annee")} : déclaré={decl:.1f}%, calculé={calc:.1f}%')

    score = max(0.0, 1.0 - len(errors) / max(len(gt.get('historique', {})) + len(res.get('tableau', [])), 1))
    return {
        'faithfulness_score': round(score, 2),
        'errors'  : errors,
        'warnings': warnings,
        'status'  : 'PASS' if not errors else 'FAIL'
    }


def test_relevance(result: dict) -> dict:
    res     = result.get('result', {})
    missing = [f for f in REQUIRED_FIELDS_T2 if not res.get(f)]
    score   = 1.0 - len(missing) / len(REQUIRED_FIELDS_T2)
    return {
        'relevance_score': round(score, 2),
        'missing_fields' : missing,
        'n_projections'  : len(res.get('tableau', [])),
        'status'         : 'PASS' if not missing else 'FAIL'
    }


print('✅ GROUND_TRUTH, test_faithfulness et test_relevance définis')

✅ GROUND_TRUTH, test_faithfulness et test_relevance définis


## 9. Questions officielles — Exécution

In [50]:
QUESTIONS = [
    {
        'label'   : "Q1 — Chiffre d'Affaires 2026-2029",
        'question': "Valide la projection du Chiffre d'Affaires FallahTech 2026-2029 "
                    "par rapport à l'historique NCT 2023-2025.",
        'gt_key'  : 'CA'
    },
    {
        'label'   : 'Q2 — Charges de personnel 2026-2028',
        'question': 'La projection des charges de personnel pour 2026-2028 '
                    'est-elle cohérente avec la dynamique historique 2023-2025 ?',
        'gt_key'  : 'Charges_Personnel'
    },
    {
        'label'   : 'Q3 — Marge brute 70% sur 2026-2029',
        'question': 'Analyse la cohérence de la marge brute projetée à 70% sur 2026-2029 '
                    'face à l\'historique certifié NCT 2023-2025.',
        'gt_key'  : 'Marge_Brute_Pct'
    },
]


def run_analysis(q_meta: dict, verbose: bool = False) -> dict:
    print(f"\n{'='*60}")
    print(f"  {q_meta['label']}")
    print(f"{'='*60}")

    result = analyser_t2(q_meta['question'], k=6, verbose=verbose)

    # [CHANGEMENT 8] Afficher un aperçu du raw en cas d'erreur JSON
    if 'error' in result.get('result', {}):
        print('\n⚠️  ERREUR PARSING JSON — Aperçu du raw output :')
        print(result.get('raw_output', '')[:800])
        print('...')

    display_t2(result)

    rel   = test_relevance(result)
    faith = test_faithfulness(result, q_meta.get('gt_key', ''))

    print(f"\n  Tokens         : {result.get('tokens','?')}")
    print(f"  Relevance      : {rel['status']} (score={rel['relevance_score']})")
    if rel['missing_fields']:
        print(f"  Champs manquants: {rel['missing_fields']}")
    print(f"  Faithfulness   : {faith['status']} (score={faith['faithfulness_score']})")
    for e in faith['errors']:
        print(f'    Erreur: {e}')
    for w in faith['warnings']:
        print(f'    Attention: {w}')

    return {'result': result, 'relevance': rel, 'faithfulness': faith}


print('✅ run_analysis défini — prêt à lancer les 3 questions')

✅ run_analysis défini — prêt à lancer les 3 questions


### Q1 — Chiffre d'Affaires 2026-2029

In [28]:
R1 = run_analysis(QUESTIONS[0])


  Q1 — Chiffre d'Affaires 2026-2029


Hypothèse,Valeur projetée,Valeur historique (réf.),Écart,Niveau de risque
Chiffre d'Affaires 2026La projection pour 2026 est considérée comme conservatrice par rapport à la croissance historique.,2 520 000 TND,1 650 000 TND (2025),+53%,Conservatrice
Chiffre d'Affaires 2027La projection pour 2027 est considérée comme optimiste par rapport à la croissance historique.,3 657 000 TND,1 650 000 TND (2025),+121%,Optimiste
Chiffre d'Affaires 2028La projection pour 2028 est considérée comme très optimiste par rapport à la croissance historique.,5 118 000 TND,1 650 000 TND (2025),+210%,très optimiste
Chiffre d'Affaires 2029La projection pour 2029 est considérée comme très optimiste par rapport à la croissance historique.,6 910 000 TND,1 650 000 TND (2025),+318%,très optimiste



  Tokens         : 4768
  Relevance      : PASS (score=1.0)
  Faithfulness   : PASS (score=1.0)


### Q2 — Charges de personnel 2026-2028

In [ ]:
R2 = run_analysis(QUESTIONS[1])


  Q2 — Charges de personnel 2026-2028


Hypothèse,Valeur projetée,Valeur historique (réf.),Écart,Niveau de risque
"Charges de personnel 2026La projection pour 2026 est inférieure à la croissance historique, ce qui indique une hypothèse conservatrice.",600 000 TND,480 000 TND (2025),25%,Conservatrice
"Charges de personnel 2027La projection pour 2027 est inférieure à la croissance historique, ce qui indique une hypothèse conservatrice.",750 000 TND,600 000 TND (2026),25%,Conservatrice
"Charges de personnel 2028La projection pour 2028 est inférieure à la croissance historique, ce qui indique une hypothèse conservatrice.",937 500 TND,750 000 TND (2027),25%,Conservatrice



  Tokens         : 7597
  Relevance      : PASS (score=1.0)
  Faithfulness   : FAIL (score=0.67)
    Erreur: Écart 2027 : déclaré=25.0%, calculé=56.2%
    Erreur: Écart 2028 : déclaré=25.0%, calculé=95.3%


### Q3 — Marge brute 70% sur 2026-2029

In [ ]:
R3 = run_analysis(QUESTIONS[2])


  Q3 — Marge brute 70% sur 2026-2029


Hypothèse,Valeur projetée,Valeur historique (réf.),Écart,Niveau de risque
"Marge brute 2026La marge brute projetée pour 2026 est identique à celle de 2025, ce qui représente une hypothèse conservatrice.",70%,70% (2025),0%,Conservatrice
"Marge brute 2027La marge brute projetée pour 2027 est identique à celle de 2025, ce qui représente une hypothèse conservatrice.",70%,70% (2025),0%,Conservatrice
"Marge brute 2028La marge brute projetée pour 2028 est identique à celle de 2025, ce qui représente une hypothèse conservatrice.",70%,70% (2025),0%,Conservatrice
"Marge brute 2029La marge brute projetée pour 2029 est identique à celle de 2025, ce qui représente une hypothèse conservatrice.",70%,70% (2025),0%,Conservatrice



  Tokens         : 6896
  Relevance      : PASS (score=1.0)
  Faithfulness   : PASS (score=1.0)


### Résumé évaluation

In [ ]:
print(f"\n{'='*60}")
print('  RÉSUMÉ ÉVALUATION T2')
print(f"{'='*60}")
print(f"{'Question':<35} {'Relev.':>7} {'Faith.':>8} {'Tokens':>7}")
print('-' * 60)
for q, r in zip(QUESTIONS, [R1, R2, R3]):
    rel_s   = r['relevance']['relevance_score']
    faith_s = r['faithfulness']['faithfulness_score']
    tok     = r['result'].get('tokens', '?')
    print(f"{q['label'][:34]:<35} {rel_s:>7.2f} {faith_s:>8.2f} {str(tok):>7}")


  RÉSUMÉ ÉVALUATION T2
Question                             Relev.   Faith.  Tokens
------------------------------------------------------------
Q1 — Chiffre d'Affaires 2026-2029      1.00     1.00    7011
Q2 — Charges de personnel 2026-202     1.00     0.67    7597
Q3 — Marge brute 70% sur 2026-2029     1.00     1.00    6896


## 8. Question libre — Réponse directe

In [39]:
MA_QUESTION = "Compare les charges entre 2023, 2024 et 2025"

print(f"Question : {MA_QUESTION}")
print('Analyse en cours...')
RES_LIBRE = analyser_t2(MA_QUESTION, k=7)
display_t2(RES_LIBRE)
print(f"Tokens consommes : {RES_LIBRE.get('tokens','?')}")




Question : Compare les charges entre 2023, 2024 et 2025
Analyse en cours...


Hypothèse,Valeur projetée,Valeur historique (réf.),Écart,Niveau de risque
"Charges d'exploitation 2026La croissance des charges d'exploitation est inférieure à la croissance historique, ce qui suggère une gestion efficace ...",1 930 000 TND,1 605 000 TND (2025),"+20,2%",Réaliste
"Charges d'exploitation 2027La croissance des charges d'exploitation est supérieure à la croissance historique, ce qui suggère une augmentation des ...",2 403 000 TND,1 605 000 TND (2025),"+49,6%",Optimiste
"Charges d'exploitation 2028La croissance des charges d'exploitation est très élevée, ce qui suggère une augmentation significative des coûts liés à...",3 002 000 TND,1 605 000 TND (2025),"+86,8%",Très optimiste
"Charges d'exploitation 2029La croissance des charges d'exploitation est très élevée, ce qui suggère une augmentation significative des coûts liés à...",3 729 000 TND,1 605 000 TND (2025),"+132,2%",Très optimiste


Tokens consommes : 8992


In [ ]:
Analyse la cohérence d’une marge brute de 70% sur 2026-2029
Analyse l’évolution de la rentabilité de l’entreprise.
Analyse les flux de trésorerie sur la période historique